In [1]:
import pandas as pd
from faker import Faker
import random
import os

# defining my path into my downloads you can change according to your preference

downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")

raw_file = os.path.join(downloads_path, "raw_crm_data.csv")
clean_file = os.path.join(downloads_path, "cleaned_crm_data.csv")
recon_file = os.path.join(downloads_path, "reconciliation_summary.csv")


fake = Faker()
records = []

for i in range(15000):
    records.append({
        "contact_id": f"CON-{i+1:05d}",
        "name": fake.name(),
        "email": fake.email() if random.random() > 0.05 else "",
        "phone": fake.msisdn() if random.random() > 0.08 else None,
        "account": fake.company(),
        "region": random.choice(["North", "South", "East", "West", "APAC"]),
        "deal_value": round(random.uniform(5000, 500000), 2),
        "stage": random.choice(["Prospect", "Qualified", "Proposal", "Closed Won", "Closed Lost"]),
        "last_activity": fake.date_between(start_date="-2y", end_date="today"),
        "owner": fake.name()
    })

df = pd.DataFrame(records)

# Add realistic issues
df = pd.concat([df, df.sample(frac=0.03, random_state=42)], ignore_index=True)
df.loc[df.sample(frac=0.01).index, "deal_value"] *= -1
df.loc[df.sample(frac=0.02).index, "stage"] = "closed won"
df.loc[df.sample(frac=0.02).index, "stage"] = "CLOSED WON"
df.loc[df.sample(frac=0.03).index, "owner"] = None

# Save RAW
df.to_csv(raw_file, index=False)
print(f" Raw data saved → {raw_file}")

 Raw data saved → C:\Users\prave\Downloads\raw_crm_data.csv


In [3]:

clean_df = df.copy()

clean_df = clean_df.drop_duplicates()
clean_df["stage"] = clean_df["stage"].str.title()
clean_df = clean_df[clean_df["deal_value"] > 0]
clean_df["owner"] = clean_df["owner"].fillna("Unassigned")
clean_df["email"] = clean_df["email"].replace("", None)

clean_df.to_csv(clean_file, index=False)
print(f" Cleaned data saved → {clean_file}")


 Cleaned data saved → C:\Users\prave\Downloads\cleaned_crm_data.csv


In [4]:
summary = pd.DataFrame({
    "Metric": [
        "Raw Rows",
        "Clean Rows",
        "Duplicates Removed",
        "Missing Emails",
        "Negative Deals"
    ],
    "Value": [
        len(df),
        len(clean_df),
        df.duplicated().sum(),
        df["email"].isnull().sum(),
        (df["deal_value"] < 0).sum()
    ]
})

summary.to_csv(recon_file, index=False)
print(f" Reconciliation report saved → {recon_file}")


 Reconciliation report saved → C:\Users\prave\Downloads\reconciliation_summary.csv


In [5]:
print("\n SUMMARY")
print(summary)


 SUMMARY
               Metric  Value
0            Raw Rows  15450
1          Clean Rows  14912
2  Duplicates Removed    384
3      Missing Emails      0
4      Negative Deals    154
